In [1]:
import pyranges
import pandas
import scanpy
import hdf5plugin
import anndata
import cellrank
import matplotlib
from matplotlib import pyplot
import cellrank
import numpy
import gseapy
import pygam
import seaborn
import pydeseq2
import pydeseq2.dds
import pydeseq2.ds
import scipy

In [2]:
# Load data from Cellrank and SDEvelo

working_directory = "RNA Sequencing Data/"
base_name = "cellrank_version_5"
adata = scanpy.read_h5ad(
    working_directory+f"/{base_name}_anndata.h5ad"
)
driver_df = pandas.read_csv(
    working_directory+f"{base_name}_driver_genes.csv",
    index_col=0
)

In [3]:
# Load adata with raw counts and filter

adata_raw = scanpy.read_h5ad(
    working_directory+f"/splice_counts_mm10.h5ad"
)

# Calculate proportion of mitochondrial genes
adata_raw.var["mt"] = adata_raw.var_names.str.startswith("mt-")
scanpy.pp.calculate_qc_metrics(
    adata_raw, qc_vars=["mt"], inplace=True, percent_top=[], log1p=False
)

scanpy.pp.calculate_qc_metrics(
    adata_raw, inplace=True, percent_top=[], log1p=False
)

# Filter cells by counts per cell, total counts and percent mitochondrial
included_cells = (adata_raw.obs["n_genes_by_counts"] >= 200)*(adata_raw.obs['total_counts'] <= 150000)*(adata_raw.obs['pct_counts_mt'] < 10)
adata_raw = adata_raw[included_cells]
print(adata_raw)

View of AnnData object with n_obs × n_vars = 12791 × 48526
    obs: 'barcode', 'batch', 'sample', 'group', 'day', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'ensemble_ids', 'gene_symbol', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'sample_colors'
    obsm: 'X_fdl', 'X_umap'
    layers: 'ambiguous', 'matrix', 'spliced', 'unspliced'


# Differential expression pseudobulk analysis with PyDEseq2

Article describing PyDESeq2: https://academic.oup.com/bioinformatics/article/39/9/btad547/7260507

Article describing DESeq2: https://link.springer.com/article/10.1186/s13059-014-0550-8

In [4]:
filtered_adata = adata_raw[:, adata.var_names].copy()
filtered_adata.X = filtered_adata.X.toarray()

In [5]:
# Combine counts within samples

sample_names = adata.obs["sample"].unique()

pseudobulk_rows = []
pseudobulk_index = []
sample_metadata_list = []

for sample in sample_names:
    cells_mask = filtered_adata.obs["sample"] == sample
    sample_X = filtered_adata.X[cells_mask]
    summed_counts = sample_X.sum(axis=0)
        
    pseudobulk_rows.append(summed_counts)
    pseudobulk_index.append(sample)
    
    # Determine the group of the sample
    group = adata.obs.loc[cells_mask, "group"].iloc[0]
    day = adata.obs.loc[cells_mask, "day"].iloc[0]
    sample_metadata_list.append({"sample": sample, "group": group, "day": day})

# Create the counts dataframe
counts_df = pandas.DataFrame(
    data=pseudobulk_rows,
    index=pseudobulk_index,
    columns=adata.var_names
)

# Create metadata dataframe
metadata_df = pandas.DataFrame(sample_metadata_list).set_index("sample")

In [6]:
# Differential expression analysis with pydeseq2

reprogramming_adata = adata[adata.obs["group"].isin({"Hic2", "control"})].copy()
inference = pydeseq2.dds.DefaultInference(n_cpus=8)
dds = pydeseq2.dds.DeseqDataSet(
    counts=counts_df,
    metadata=metadata_df,
    design="~group",
    refit_cooks=True,
    inference=inference
)

# Fit model
dds.deseq2()


Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.32 seconds.

Fitting dispersion trend curve...
... done in 0.04 seconds.

Fitting MAP dispersions...
... done in 0.38 seconds.

Fitting LFCs...
... done in 0.22 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.



In [7]:
stat_results = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "control"], inference=inference)

# compute p-values and adjusted p-values
stat_results.summary()

Log2 fold change & Wald test p-value: group Hic2 vs control
                 baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_names                                                                
Rb1cc1        2239.163976       -0.283591  0.244634 -1.159246  0.246356   
Fam150a         33.706046        0.335673  1.152714  0.291203  0.770896   
Prex2          155.234879        1.573466  0.503496  3.125081  0.001778   
Sulf1          677.858231        0.077258  0.591976  0.130508  0.896164   
Msc            423.997047        2.369996  0.716943  3.305699  0.000947   
...                   ...             ...       ...       ...       ...   
Egfl6          166.625885        0.470281  0.781868  0.601483  0.547518   
Tmsb4x      115027.155006        0.654934  0.384154  1.704876  0.088218   
Usp9y           21.430362        2.368054  1.032823  2.292798  0.021860   
Erdr1         1123.037550        0.974630  0.549626  1.773260  0.076186   
AC168977.1       7.812825       -0.62421

Running Wald tests...
... done in 0.12 seconds.



In [8]:
# Whether genes of interest are up- or downregulated

stem_cell_tfs = ['Jarid2', 'Klf2', 'Mycn', 'Rest', 'Sall1', 'Tcf7l1', 'Tcf7l2',
       'Tfcp2l1', 'Tsc22d1', 'Zfp42']
dead_end_tfs = ['Barx2', 'Dmrtc2', 'Hmgb3', 'Hoxa7', 'Myc', 'Ovol1', 'Phox2a',
       'Tlx2', 'Tshz1', 'Zbtb7c']
genes_of_interest = stem_cell_tfs + dead_end_tfs
stat_results.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_names,,,,,,
Ovol1,340.259652,-3.484112,0.724528,-4.808799,0.000002,0.000080
Tlx2,762.591789,-3.304927,0.750462,-4.403856,0.000011,0.000357
Dmrtc2,884.994216,-3.374380,0.829364,-4.068636,0.000047,0.001226
Hmgb3,1503.378041,-1.429928,0.386190,-3.702652,0.000213,0.004153
Zbtb7c,747.489340,-1.675318,0.482144,-3.474728,0.000511,0.008153
Barx2,241.839421,-2.414571,0.841142,-2.870587,0.004097,0.040247
Phox2a,449.720898,-1.572235,0.579984,-2.710826,0.006712,0.056524
Tcf7l1,1565.285438,1.031554,0.439377,2.347767,0.018886,0.115094
Myc,2758.170812,-0.940123,0.450300,-2.087771,0.036818,0.167710


In [9]:
# Most upregulated in control

stat_results.results_df.query("padj < 0.05 and log2FoldChange < 0").sort_values("log2FoldChange").head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_names,,,,,,
2310046K23Rik,37.150773,-8.101822,1.859812,-4.356259,1.323045e-05,4.361563e-04
Padi4,81.129161,-5.525528,0.887838,-6.223574,4.859564e-10,9.451853e-08
Mal2,397.453478,-5.088056,0.869783,-5.849797,4.921734e-09,7.363671e-07
Vsig8,70.732354,-4.959581,0.871917,-5.688136,1.284334e-08,1.561268e-06
Prss32,1024.286463,-4.922016,0.986613,-4.988802,6.075483e-07,3.938938e-05
Krt14,1462.170985,-4.686814,0.675246,-6.940899,3.896137e-12,1.262998e-09
Ly6g6c,5682.670391,-4.670438,0.962784,-4.850971,1.228587e-06,6.827435e-05
Lgals7,9929.055723,-4.641645,0.854930,-5.429268,5.658576e-08,4.785187e-06
Pkp1,401.631709,-4.622098,0.650179,-7.108967,1.169150e-12,4.547994e-10


In [10]:
# Most upregulated in Hic2

stat_results.results_df.query("padj < 0.05 and log2FoldChange > 0").sort_values("log2FoldChange", ascending=False).head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_names,,,,,,
Xist,792.024515,3.977165,1.005491,3.955448,7.639151e-05,1.857269e-03
Cd36,25.023779,3.873931,1.143431,3.387989,7.040699e-04,1.029636e-02
Marc1,24.456462,3.576419,0.892136,4.008825,6.102151e-05,1.522021e-03
Stra8,89.375272,3.541480,1.127707,3.140425,1.687032e-03,2.089985e-02
Xirp2,83.190523,3.082889,0.872794,3.532207,4.121060e-04,6.909881e-03
Cdh5,19.222726,3.024164,1.058710,2.856462,4.283913e-03,4.166105e-02
Spink6,436.255228,2.897080,0.983585,2.945431,3.225055e-03,3.465597e-02
Myl3,35.093383,2.895067,0.861486,3.360551,7.778707e-04,1.119690e-02
Bhmt,39.421950,2.885319,1.006768,2.865921,4.157980e-03,4.063955e-02


# Differential expression in bulk data with PyDEseq2

Article describing PyDESeq2: https://academic.oup.com/bioinformatics/article/39/9/btad547/7260507

Article describing DESeq2: https://link.springer.com/article/10.1186/s13059-014-0550-8

In [11]:
bulk_counts = pandas.read_csv("RNA Sequencing Data/Bulk RNA sequencing data Hic2/counts2.tsv", sep="\t", index_col=0).T
samples = bulk_counts.index.to_list()
print(samples)

['Day2_BFP_Rep1', 'Day2_BFP_Rep2', 'Day2_BFP_Rep3', 'Day2_Hic2_Rep1', 'Day2_Hic2_Rep2', 'Day2_Hic2_Rep3', 'Day4_BFP_Rep1', 'Day4_BFP_Rep1-2', 'Day4_BFP_Rep2', 'Day4_BFP_Rep3', 'Day4_Hic2_Rep1', 'Day4_Hic2_Rep1-2', 'Day4_Hic2_Rep2', 'Day4_Hic2_Rep3', 'Day6_BFP_Rep1', 'Day6_BFP_Rep2', 'Day6_BFP_Rep3', 'Day6_Hic2_Rep1', 'Day6_Hic2_Rep2', 'Day6_Hic2_Rep3', 'Day10_BFP_Rep1', 'Day10_BFP_Rep2', 'Day10_BFP_Rep3', 'Day10_Hic2_Rep1', 'Day10_Hic2_Rep2', 'Day10_Hic2_Rep3', 'ESC_BFP_Rep1', 'ESC_BFP_Rep2', 'ESC_BFP_Rep3', 'ESC_Hic2-gRNA_Rep1', 'ESC_Hic2-gRNA_Rep2', 'ESC_Hic2-gRNA_Rep3', 'ESC_Zeo-gRNA_Rep1', 'ESC_Zeo-gRNA_Rep2', 'ESC_Zeo-gRNA_Rep3', 'iPSC_BFP_Rep1', 'iPSC_BFP_Rep2', 'iPSC_BFP_Rep3', 'iPSC_Hic2_Rep1', 'iPSC_Hic2_Rep2', 'iPSC_Hic2_Rep3', 'MEF_BFP_Rep1', 'MEF_BFP_Rep2', 'MEF_BFP_Rep3', 'MEF_Hic2_Rep1', 'MEF_Hic2_Rep2', 'MEF_Hic2_Rep3']


In [12]:
# Create metadata dataframe

sample_metadata_list = []

for sample in samples:
    sample_name_list = sample.split("_")
    if ("Day" in sample_name_list[0]):
        day = int(sample_name_list[0][3:])
        experiment = "reprogramming"
    else:
        day = 0
        experiment = sample_name_list[0]
    group = sample_name_list[1]
    replicate_list = sample_name_list[2][3:].split("-")
    replicate = replicate_list[0]
    if len(replicate_list) == 2:
        batch = int(replicate_list[1])
    else:
        batch = 1 if experiment in {"reprogramming", "MEF"} else 2
    
    sample_metadata_list.append({"sample": sample, "day": day, "experiment": experiment, "group": group, "replicate": replicate, "batch": batch})

# Create metadata dataframe
metadata_df = pandas.DataFrame(sample_metadata_list).set_index("sample")

metadata_df

,day,experiment,group,replicate,batch
sample,,,,,
Day2_BFP_Rep1,2,reprogramming,BFP,1,1
Day2_BFP_Rep2,2,reprogramming,BFP,2,1
Day2_BFP_Rep3,2,reprogramming,BFP,3,1
Day2_Hic2_Rep1,2,reprogramming,Hic2,1,1
Day2_Hic2_Rep2,2,reprogramming,Hic2,2,1
Day2_Hic2_Rep3,2,reprogramming,Hic2,3,1
Day4_BFP_Rep1,4,reprogramming,BFP,1,1
Day4_BFP_Rep1-2,4,reprogramming,BFP,1,2
Day4_BFP_Rep2,4,reprogramming,BFP,2,1


In [13]:
# Fix duplicated columns
duplicated_columns = bulk_counts.columns[bulk_counts.columns.duplicated()].to_list()
print(duplicated_columns)

# These likely result from Excel misinterpreting gene names as dates, possibly MARC1 and MARC2. They can be ignored.
bulk_counts = bulk_counts.drop(columns=duplicated_columns)

['Mar-01', 'Mar-02']


In [14]:
# Find columns that are also in the adata object

variable_genes = adata.var.index.to_numpy()
common_genes = numpy.intersect1d(variable_genes, bulk_counts.columns.to_numpy())

## Differential expression during the reprogramming experiment

In [15]:
reprogramming_samples = metadata_df.query("experiment == \"reprogramming\"").index.to_list()
print(metadata_df.loc[reprogramming_samples])

                  day     experiment group replicate  batch
sample                                                     
Day2_BFP_Rep1       2  reprogramming   BFP         1      1
Day2_BFP_Rep2       2  reprogramming   BFP         2      1
Day2_BFP_Rep3       2  reprogramming   BFP         3      1
Day2_Hic2_Rep1      2  reprogramming  Hic2         1      1
Day2_Hic2_Rep2      2  reprogramming  Hic2         2      1
Day2_Hic2_Rep3      2  reprogramming  Hic2         3      1
Day4_BFP_Rep1       4  reprogramming   BFP         1      1
Day4_BFP_Rep1-2     4  reprogramming   BFP         1      2
Day4_BFP_Rep2       4  reprogramming   BFP         2      1
Day4_BFP_Rep3       4  reprogramming   BFP         3      1
Day4_Hic2_Rep1      4  reprogramming  Hic2         1      1
Day4_Hic2_Rep1-2    4  reprogramming  Hic2         1      2
Day4_Hic2_Rep2      4  reprogramming  Hic2         2      1
Day4_Hic2_Rep3      4  reprogramming  Hic2         3      1
Day6_BFP_Rep1       6  reprogramming   B

In [16]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[reprogramming_samples],
    metadata=metadata_df.loc[reprogramming_samples],
    design="~C(day) + C(batch) + group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
reprogramming_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
reprogramming_de_stats.summary()

Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.04 seconds.

Fitting dispersions...
... done in 2.22 seconds.

Fitting dispersion trend curve...
... done in 0.51 seconds.

Fitting MAP dispersions...
... done in 2.02 seconds.

Fitting LFCs...
... done in 2.30 seconds.

Calculating cook's distance...
... done in 0.05 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                baseMean  log2FoldChange     lfcSE      stat        pvalue  \
Gnai3       18118.223009       -0.237734  0.046419 -5.121532  3.030633e-07   
Pbsn            0.000000             NaN       NaN       NaN           NaN   
Cdc45        1046.343936        0.800273  0.094942  8.429116  3.482852e-17   
H19          4548.672607        0.729453  0.114614  6.364408  1.960450e-10   
Scml2         183.121673       -0.602971  0.144454 -4.174135  2.991209e-05   
...                  ...             ...       ...       ...           ...   
BX571804.1      0.000000             NaN       NaN       NaN           NaN   
AC154773.1      0.000000             NaN       NaN       NaN           NaN   
AL662853.1      0.286036        0.360430  3.104910  0.116084  9.075861e-01   
AC145556.1      0.781459       -1.160610  1.788140 -0.649060  5.162997e-01   
CT868734.1      0.000000             NaN       NaN       NaN           NaN   

       

... done in 1.89 seconds.



In [18]:
# Whether genes of interest are up- or downregulated

reprogramming_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Zbtb7c,1521.129230,-0.943320,0.058151,-16.221943,3.528526e-59,8.875744e-57
Sall1,914.264618,1.516870,0.152381,9.954437,2.411844e-23,8.706573e-22
Barx2,270.806735,-1.207086,0.122707,-9.837179,7.786359e-23,2.699537e-21
Zfp42,2688.373406,1.344900,0.149763,8.980156,2.703821e-19,6.666513e-18
Hmgb3,1523.258054,-0.857015,0.103484,-8.281647,1.214838e-16,2.360299e-15
Ovol1,680.945017,-1.451013,0.185601,-7.817914,5.370601e-15,8.644510e-14
Tcf7l1,582.207950,1.263931,0.164272,7.694150,1.424378e-14,2.177080e-13
Tlx2,585.037236,-1.107785,0.157145,-7.049422,1.796624e-12,2.177406e-11
Tfcp2l1,2321.616881,1.502553,0.225386,6.666573,2.618455e-11,2.759063e-10
Tcf7l2,1162.635586,0.753992,0.117507,6.416592,1.393588e-10,1.340578e-09


In [19]:
# Higher expression in control

reprogramming_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange < 0").sort_values("log2FoldChange").head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
2310046K23Rik,121.261593,-5.026695,0.526483,-9.547682,1.326310e-21,4.142749e-20
Krt14,5110.427482,-4.106556,0.314381,-13.062360,5.402643e-39,5.300643e-37
Krt6a,8104.897532,-3.767656,0.231002,-16.310079,8.368502e-60,2.198591e-57
Mlc1,6.222436,-3.765394,0.772700,-4.873032,1.098985e-06,6.230041e-06
Slurp1,817.514135,-3.661155,0.243806,-15.016651,5.712186e-51,9.931224e-49
Sel1l3,35.552444,-3.592158,0.570257,-6.299195,2.991958e-10,2.773220e-09
Chit1,745.571491,-3.592043,0.181281,-19.814787,2.219452e-87,2.099158e-84
Mmp13,74.221433,-3.585201,0.291299,-12.307646,8.239182e-35,6.537431e-33
Krt17,5304.148465,-3.505668,0.151732,-23.104406,4.181106e-118,1.098469e-114
Fam180a,50.590588,-3.467625,0.341159,-10.164249,2.863197e-24,1.124589e-22


The high upregulation of 2310046K23Rik, also known as Small proline rich protein 5 (Sprr5) is very interesting.

Function of human Sprr5 according to Uniprot: "Positively regulates keratinocyte differentiation by inducing genes associated with epidermal differentiation".

In [20]:
# Higher expression with Hic2

reprogramming_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange > 0").sort_values("log2FoldChange", ascending=False).head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Ifi27l2a,60.427621,4.964874,0.849305,5.845811,5.041075e-09,3.957378e-08
Eras,129.003245,4.766440,0.745645,6.392374,1.633294e-10,1.556959e-09
3830417A13Rik,274.669327,4.546757,0.306071,14.855229,6.434730e-50,1.071473e-47
Oasl2,329.182784,4.419285,0.438522,10.077691,6.933599e-24,2.609246e-22
Pramef12,591.836100,4.304483,0.708941,6.071709,1.265565e-09,1.073508e-08
Dnmt3l,2023.817274,4.146114,0.878535,4.719352,2.365970e-06,1.268270e-05
Nr0b1,399.205895,4.031501,0.463751,8.693253,3.522062e-18,7.916269e-17
Napsa,4.379697,4.014911,1.029909,3.898317,9.686348e-05,3.921139e-04
Cdh5,11.450055,3.893308,0.550767,7.068885,1.561839e-12,1.900653e-11
Gm13119,27.830769,3.853006,0.458546,8.402669,4.364452e-17,8.934847e-16


## Differential expression in day 2

In [21]:
day_2_samples = metadata_df.query("experiment == \"reprogramming\" and day == 2").index.to_list()
print(metadata_df.loc[day_2_samples])

                day     experiment group replicate  batch
sample                                                   
Day2_BFP_Rep1     2  reprogramming   BFP         1      1
Day2_BFP_Rep2     2  reprogramming   BFP         2      1
Day2_BFP_Rep3     2  reprogramming   BFP         3      1
Day2_Hic2_Rep1    2  reprogramming  Hic2         1      1
Day2_Hic2_Rep2    2  reprogramming  Hic2         2      1
Day2_Hic2_Rep3    2  reprogramming  Hic2         3      1


In [22]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[day_2_samples],
    metadata=metadata_df.loc[day_2_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
day_2_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
day_2_de_stats.summary()

Fitting size factors...
... done in 0.02 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 1.23 seconds.

Fitting dispersion trend curve...
... done in 0.47 seconds.

Fitting MAP dispersions...
... done in 1.42 seconds.

Fitting LFCs...
... done in 1.80 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                baseMean  log2FoldChange     lfcSE      stat        pvalue  \
Gnai3       22128.672245       -0.120252  0.037410 -3.214455  1.306925e-03   
Pbsn            0.000000             NaN       NaN       NaN           NaN   
Cdc45         690.247113        0.478694  0.089751  5.333594  9.628743e-08   
H19          2598.156315        0.973740  0.318548  3.056811  2.237051e-03   
Scml2         286.322931       -0.440445  0.153644 -2.866664  4.148229e-03   
...                  ...             ...       ...       ...           ...   
BX571804.1      0.000000             NaN       NaN       NaN           NaN   
AC154773.1      0.000000             NaN       NaN       NaN           NaN   
AL662853.1      0.000000             NaN       NaN       NaN           NaN   
AC145556.1      1.836549       -1.380029  2.047491 -0.674010  5.003051e-01   
CT868734.1      0.000000             NaN       NaN       NaN           NaN   

       

... done in 2.06 seconds.



In [23]:
# Whether genes of interest are up- or downregulated

day_2_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Zbtb7c,1545.528898,-0.915841,0.073941,-12.386172,3.105169e-35,2.831547e-33
Tshz1,2793.118088,0.592365,0.057745,10.258316,1.085839e-24,5.497585e-23
Tlx2,536.024936,-1.211808,0.131036,-9.247933,2.288763e-20,8.782740e-19
Ovol1,331.654674,-1.328884,0.146448,-9.074094,1.146274e-19,4.084445e-18
Sall1,384.567579,1.171922,0.130355,8.990206,2.467675e-19,8.559147e-18
Hmgb3,1901.452329,-0.379427,0.067962,-5.582933,2.364953e-08,2.756836e-07
Tcf7l1,469.488292,0.578413,0.116473,4.966068,6.832397e-07,6.384809e-06
Jarid2,1936.496621,0.309273,0.064191,4.817978,1.450205e-06,1.280423e-05
Rest,3240.580307,0.212969,0.063922,3.331709,8.631454e-04,4.206842e-03
Tcf7l2,1152.934163,0.308370,0.097661,3.157558,1.590966e-03,7.230541e-03


## Differential expression in day 4

In [26]:
day_4_samples = metadata_df.query("experiment == \"reprogramming\" and day == 4").index.to_list()
print(metadata_df.loc[day_4_samples])

                  day     experiment group replicate  batch
sample                                                     
Day4_BFP_Rep1       4  reprogramming   BFP         1      1
Day4_BFP_Rep1-2     4  reprogramming   BFP         1      2
Day4_BFP_Rep2       4  reprogramming   BFP         2      1
Day4_BFP_Rep3       4  reprogramming   BFP         3      1
Day4_Hic2_Rep1      4  reprogramming  Hic2         1      1
Day4_Hic2_Rep1-2    4  reprogramming  Hic2         1      2
Day4_Hic2_Rep2      4  reprogramming  Hic2         2      1
Day4_Hic2_Rep3      4  reprogramming  Hic2         3      1


In [27]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[day_4_samples],
    metadata=metadata_df.loc[day_4_samples],
    design="~C(day) + C(batch) + group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
day_4_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
day_4_de_stats.summary()

Fitting size factors...
... done in 0.02 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 1.43 seconds.

Fitting dispersion trend curve...
... done in 0.48 seconds.

Fitting MAP dispersions...
... done in 1.52 seconds.

Fitting LFCs...
... done in 1.82 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                baseMean  log2FoldChange     lfcSE      stat        pvalue  \
Gnai3       15061.034487       -0.145678  0.048888 -2.979812  2.884256e-03   
Pbsn            0.000000             NaN       NaN       NaN           NaN   
Cdc45         695.773380        0.949563  0.116376  8.159410  3.366663e-16   
H19          3270.342112        0.798852  0.092818  8.606611  7.525230e-18   
Scml2         239.054249       -0.541207  0.135172 -4.003823  6.232701e-05   
...                  ...             ...       ...       ...           ...   
BX571804.1      0.000000             NaN       NaN       NaN           NaN   
AC154773.1      0.000000             NaN       NaN       NaN           NaN   
AL662853.1      0.863819        1.292333  2.205131  0.586057  5.578371e-01   
AC145556.1      0.339300        0.601040  3.320829  0.180991  8.563748e-01   
CT868734.1      0.000000             NaN       NaN       NaN           NaN   

       

... done in 1.97 seconds.



In [28]:
# Whether genes of interest are up- or downregulated

day_4_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Sall1,860.094175,1.705273,0.072756,23.438206,1.743830e-121,4.738640e-119
Zbtb7c,1758.713380,-1.095422,0.053464,-20.488793,2.710318e-93,3.954336e-91
Zfp42,427.157105,1.555553,0.137043,11.350841,7.345297e-30,1.878581e-28
Ovol1,796.688716,-0.757347,0.083485,-9.071703,1.171713e-19,1.685762e-18
Tcf7l1,356.692664,0.890082,0.107501,8.279757,1.234273e-16,1.398221e-15
Barx2,216.086538,-1.102030,0.133419,-8.259913,1.457781e-16,1.641155e-15
Tfcp2l1,213.209667,1.694083,0.213364,7.939867,2.023974e-15,2.080339e-14
Jarid2,2093.237750,0.371469,0.057566,6.452891,1.097362e-10,7.454858e-10
Tlx2,726.747089,-0.622547,0.103229,-6.030721,1.632299e-09,9.887027e-09
Hmgb3,1424.268916,-0.524054,0.092642,-5.656765,1.542528e-08,8.474356e-08


## Differential expression in day 6

In [31]:
day_6_samples = metadata_df.query("experiment == \"reprogramming\" and day == 6").index.to_list()
print(metadata_df.loc[day_6_samples])

                day     experiment group replicate  batch
sample                                                   
Day6_BFP_Rep1     6  reprogramming   BFP         1      1
Day6_BFP_Rep2     6  reprogramming   BFP         2      1
Day6_BFP_Rep3     6  reprogramming   BFP         3      1
Day6_Hic2_Rep1    6  reprogramming  Hic2         1      1
Day6_Hic2_Rep2    6  reprogramming  Hic2         2      1
Day6_Hic2_Rep3    6  reprogramming  Hic2         3      1


In [32]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[day_6_samples],
    metadata=metadata_df.loc[day_6_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
day_6_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
day_6_de_stats.summary()

Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.01 seconds.

Fitting dispersions...
... done in 1.43 seconds.

Fitting dispersion trend curve...
... done in 0.45 seconds.

Fitting MAP dispersions...
... done in 1.36 seconds.

Fitting LFCs...
... done in 1.69 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                baseMean  log2FoldChange     lfcSE      stat        pvalue  \
Gnai3       22363.371052       -0.307583  0.093112 -3.303380  9.552682e-04   
Pbsn            0.000000             NaN       NaN       NaN           NaN   
Cdc45        1137.191588        1.182272  0.128137  9.226605  2.793445e-20   
H19          3896.764956        0.465219  0.336784  1.381358  1.671688e-01   
Scml2          82.045704       -1.696386  0.424089 -4.000073  6.332304e-05   
...                  ...             ...       ...       ...           ...   
BX571804.1      0.000000             NaN       NaN       NaN           NaN   
AC154773.1      0.000000             NaN       NaN       NaN           NaN   
AL662853.1      0.000000             NaN       NaN       NaN           NaN   
AC145556.1      1.191800       -3.722858  4.235882 -0.878886  3.794630e-01   
CT868734.1      0.000000             NaN       NaN       NaN           NaN   

       

... done in 1.97 seconds.



In [33]:
# Whether genes of interest are up- or downregulated

day_6_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Tfcp2l1,2117.184424,2.836544,0.194685,14.569883,4.366368e-48,9.687480e-46
Sall1,907.920856,2.474940,0.207610,11.921127,9.185780e-33,9.132071e-31
Hmgb3,1381.829875,-1.399331,0.131310,-10.656667,1.623102e-26,1.028888e-24
Zfp42,1517.458376,1.986789,0.195344,10.170728,2.678989e-24,1.441978e-22
Tcf7l1,432.475319,2.128565,0.227501,9.356281,8.259261e-21,3.560681e-19
Jarid2,950.743533,1.499336,0.176889,8.476130,2.328088e-17,7.436864e-16
Ovol1,1018.742043,-1.117075,0.136306,-8.195337,2.498920e-16,7.170798e-15
Tcf7l2,1105.221422,1.159597,0.143769,8.065703,7.281554e-16,2.007171e-14
Barx2,405.059500,-1.650595,0.232617,-7.095765,1.286382e-12,2.571775e-11
Hoxa7,313.793281,-1.449888,0.251804,-5.758012,8.511049e-09,1.033655e-07


## Day 10 differential expression

In [36]:
day_10_samples = metadata_df.query("experiment == \"reprogramming\" and day == 10").index.to_list()
print(metadata_df.loc[day_10_samples])

                 day     experiment group replicate  batch
sample                                                    
Day10_BFP_Rep1    10  reprogramming   BFP         1      1
Day10_BFP_Rep2    10  reprogramming   BFP         2      1
Day10_BFP_Rep3    10  reprogramming   BFP         3      1
Day10_Hic2_Rep1   10  reprogramming  Hic2         1      1
Day10_Hic2_Rep2   10  reprogramming  Hic2         2      1
Day10_Hic2_Rep3   10  reprogramming  Hic2         3      1


In [37]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[day_10_samples],
    metadata=metadata_df.loc[day_10_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
day_10_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
day_10_de_stats.summary()

Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.01 seconds.

Fitting dispersions...
... done in 1.41 seconds.

Fitting dispersion trend curve...
... done in 0.47 seconds.

Fitting MAP dispersions...
... done in 1.34 seconds.

Fitting LFCs...
... done in 1.67 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Gnai3       14599.801326       -0.486742  0.110552 -4.402836  0.000011   
Pbsn            0.000000             NaN       NaN       NaN       NaN   
Cdc45        1823.385673        0.472664  0.184033  2.568364  0.010218   
H19          9032.655059        0.583415  0.142092  4.105908  0.000040   
Scml2         103.792330        0.022729  0.348827  0.065159  0.948047   
...                  ...             ...       ...       ...       ...   
BX571804.1      0.000000             NaN       NaN       NaN       NaN   
AC154773.1      0.000000             NaN       NaN       NaN       NaN   
AL662853.1      0.000000             NaN       NaN       NaN       NaN   
AC145556.1      0.000000             NaN       NaN       NaN       NaN   
CT868734.1      0.000000             NaN       NaN       NaN       NaN   

                padj  
Gnai3       0.000073  
Pbsn     

... done in 1.94 seconds.



In [38]:
# Whether genes of interest are up- or downregulated

day_10_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Ovol1,523.721234,-3.065469,0.268917,-11.399329,4.213554e-30,6.988580e-28
Hmgb3,1472.708056,-1.320122,0.129233,-10.215083,1.697342e-24,1.507255e-22
Hoxa7,313.478151,-2.416212,0.312123,-7.741205,9.847861e-15,3.224090e-13
Tsc22d1,5885.198478,0.899993,0.124330,7.238765,4.527873e-13,1.159055e-11
Tlx2,203.466775,-2.257631,0.325591,-6.933944,4.092656e-12,8.966760e-11
Zbtb7c,1304.171701,-1.035031,0.166461,-6.217862,5.039730e-10,7.762267e-09
Tcf7l2,855.164257,1.246685,0.210027,5.935833,2.923581e-09,3.977853e-08
Jarid2,3614.193957,1.191620,0.206203,5.778867,7.520533e-09,9.462670e-08
Barx2,362.418536,-1.480871,0.260520,-5.684284,1.313622e-08,1.579573e-07
Myc,4748.397004,-0.749060,0.137089,-5.464030,4.654450e-08,5.109965e-07


## Differential expression in MEFs

In [112]:
mef_samples = metadata_df.query("experiment == \"MEF\"").index.to_list()
print(metadata_df.loc[mef_samples])

               day experiment group replicate  batch
sample                                              
MEF_BFP_Rep1     0        MEF   BFP         1      1
MEF_BFP_Rep2     0        MEF   BFP         2      1
MEF_BFP_Rep3     0        MEF   BFP         3      1
MEF_Hic2_Rep1    0        MEF  Hic2         1      1
MEF_Hic2_Rep2    0        MEF  Hic2         2      1
MEF_Hic2_Rep3    0        MEF  Hic2         3      1


In [53]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=8)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[mef_samples, common_genes],
    metadata=metadata_df.loc[mef_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
mef_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
mef_de_stats.summary()

Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.19 seconds.

Fitting dispersion trend curve...
... done in 0.04 seconds.

Fitting MAP dispersions...
... done in 0.22 seconds.

Fitting LFCs...
... done in 0.25 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.17 seconds.



Log2 fold change & Wald test p-value: group Hic2 vs BFP
                  baseMean  log2FoldChange     lfcSE      stat        pvalue  \
0610040J01Rik    21.265536       -0.403483  0.497757 -0.810603  4.175938e-01   
1110008L16Rik   324.582825        0.239663  0.203499  1.177710  2.389124e-01   
1110038B12Rik  1234.185766        0.028475  0.202006  0.140961  8.879011e-01   
1500009L16Rik   289.683772        0.891146  0.162259  5.492105  3.971718e-08   
1500015O10Rik    71.716345       -1.477486  0.287123 -5.145826  2.663468e-07   
...                    ...             ...       ...       ...           ...   
Zmiz1          6521.568472        0.028579  0.138333  0.206597  8.363244e-01   
Zp3               0.667739       -2.910684  3.596741 -0.809256  4.183680e-01   
Zscan4a           0.000000             NaN       NaN       NaN           NaN   
Zscan4c           0.000000             NaN       NaN       NaN           NaN   
Zscan4e           0.000000             NaN       NaN       NaN  

In [54]:
# Whether genes of interest are up- or downregulated

mef_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Tcf7l2,1461.158516,0.411018,0.150840,2.724854,0.006433,0.026791
Hoxa7,806.661901,0.347961,0.135321,2.571375,0.010130,0.038594
Zbtb7c,124.670887,-0.638847,0.262957,-2.429475,0.015121,0.051838
Klf2,196.442771,-0.859362,0.378279,-2.271767,0.023101,0.072763
Barx2,38.210838,1.025243,0.494220,2.074465,0.038036,0.105690
Tfcp2l1,51.991721,-0.552028,0.295294,-1.869417,0.061565,0.152600
Tshz1,5079.950345,0.259353,0.145896,1.777661,0.075460,0.178275
Sall1,197.029590,0.269536,0.172807,1.559752,0.118818,0.252543
Tcf7l1,1334.422051,0.255627,0.184698,1.384022,0.166352,0.321822
Hmgb3,1384.007209,0.229583,0.172175,1.333433,0.182390,0.345154


## Differential expression in ESCs

In [113]:
esc_samples = metadata_df.query("experiment == \"ESC\"").index.to_list()
print(metadata_df.loc[esc_samples])

                    day experiment      group replicate  batch
sample                                                        
ESC_BFP_Rep1          0        ESC        BFP         1      2
ESC_BFP_Rep2          0        ESC        BFP         2      2
ESC_BFP_Rep3          0        ESC        BFP         3      2
ESC_Hic2-gRNA_Rep1    0        ESC  Hic2-gRNA         1      2
ESC_Hic2-gRNA_Rep2    0        ESC  Hic2-gRNA         2      2
ESC_Hic2-gRNA_Rep3    0        ESC  Hic2-gRNA         3      2
ESC_Zeo-gRNA_Rep1     0        ESC   Zeo-gRNA         1      2
ESC_Zeo-gRNA_Rep2     0        ESC   Zeo-gRNA         2      2
ESC_Zeo-gRNA_Rep3     0        ESC   Zeo-gRNA         3      2


In [115]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=8)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[esc_samples, common_genes],
    metadata=metadata_df.loc[esc_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
esc_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Zeo-gRNA", "Hic2-gRNA"], inference=inference)
# This is a knockout experiments so the order of samples is reversed
esc_de_stats.summary()

Fitting size factors...
... done in 0.00 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 0.26 seconds.

Fitting dispersion trend curve...
... done in 0.07 seconds.

Fitting MAP dispersions...
... done in 0.30 seconds.

Fitting LFCs...
... done in 0.20 seconds.



Log2 fold change & Wald test p-value: group Zeo-gRNA vs Hic2-gRNA
                  baseMean  log2FoldChange     lfcSE      stat        pvalue  \
0610040J01Rik    51.921452       -0.527384  0.290160 -1.817561  6.913131e-02   
1110008L16Rik   722.601741        0.083498  0.148153  0.563595  5.730299e-01   
1110038B12Rik  1389.865872        0.755943  0.127443  5.931615  2.999698e-09   
1500009L16Rik   154.560178        0.125743  0.267273  0.470467  6.380211e-01   
1500015O10Rik     5.207505       -3.085208  1.198319 -2.574613  1.003523e-02   
...                    ...             ...       ...       ...           ...   
Zmiz1           785.722730       -1.124409  0.246163 -4.567739  4.930129e-06   
Zp3              97.979012        0.822176  0.307450  2.674180  7.491220e-03   
Zscan4a         177.497754       -0.543751  0.570026 -0.953905  3.401316e-01   
Zscan4c         205.143733       -0.600216  0.857092 -0.700293  4.837443e-01   
Zscan4e         210.310155       -0.010496  0.606083 -

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.11 seconds.



In [116]:
# Whether genes of interest are up- or downregulated

esc_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Myc,883.743218,3.686952,0.161470,22.833730,2.120667e-115,5.225930e-113
Tfcp2l1,7033.005586,2.041081,0.153019,13.338782,1.376903e-40,7.917192e-39
Zfp42,12778.669838,1.464963,0.128270,11.420961,3.285796e-30,1.111372e-28
Rest,6352.040976,0.464189,0.076798,6.044300,1.500605e-09,1.023140e-08
Mycn,4152.026270,-1.605242,0.297809,-5.390168,7.039168e-08,3.891848e-07
Hoxa7,13.261125,2.789823,0.685144,4.071877,4.663576e-05,1.665563e-04
Hmgb3,626.869728,-0.439919,0.110975,-3.964125,7.366551e-05,2.536387e-04
Sall1,7012.419627,-0.270905,0.075207,-3.602135,3.156144e-04,9.704719e-04
Dmrtc2,44.013304,-1.193261,0.374995,-3.182070,1.462266e-03,3.935115e-03
Tcf7l1,2084.262946,0.484475,0.157901,3.068215,2.153415e-03,5.611239e-03


## Differential expresion in iPSCs

In [103]:
ipsc_samples = metadata_df.query("experiment == \"iPSC\"").index.to_list()
print(metadata_df.loc[ipsc_samples])

                day experiment group replicate  batch
sample                                               
iPSC_BFP_Rep1     0       iPSC   BFP         1      2
iPSC_BFP_Rep2     0       iPSC   BFP         2      2
iPSC_BFP_Rep3     0       iPSC   BFP         3      2
iPSC_Hic2_Rep1    0       iPSC  Hic2         1      2
iPSC_Hic2_Rep2    0       iPSC  Hic2         2      2
iPSC_Hic2_Rep3    0       iPSC  Hic2         3      2


In [104]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[ipsc_samples, common_genes],
    metadata=metadata_df.loc[ipsc_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
ipsc_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
ipsc_de_stats.summary()

Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.16 seconds.

Fitting dispersion trend curve...
... done in 0.04 seconds.

Fitting MAP dispersions...
... done in 0.18 seconds.

Fitting LFCs...
... done in 0.20 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                  baseMean  log2FoldChange     lfcSE      stat    pvalue  \
0610040J01Rik    26.488640        0.462650  0.436792  1.059199  0.289509   
1110008L16Rik   729.462893        0.112113  0.186731  0.600398  0.548241   
1110038B12Rik  1456.620150       -0.303012  0.274482 -1.103940  0.269619   
1500009L16Rik   174.852541        0.036534  0.311481  0.117293  0.906628   
1500015O10Rik     5.495177        0.981877  1.028843  0.954350  0.339906   
...                    ...             ...       ...       ...       ...   
Zmiz1           846.252400       -0.033096  0.378258 -0.087497  0.930277   
Zp3              83.709293       -0.156560  0.549123 -0.285110  0.775560   
Zscan4a         179.665409        0.237048  0.545430  0.434607  0.663847   
Zscan4c         221.557134       -0.104361  0.609409 -0.171250  0.864028   
Zscan4e         276.388521        0.371044  0.624233  0.594400  0.552244   

                   padj  
06100

... done in 0.19 seconds.



In [105]:
# Whether genes of interest are up- or downregulated

ipsc_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Jarid2,17850.319954,-0.088999,0.213711,-0.416446,0.677083,0.999831
Klf2,3946.622489,-0.417831,0.203246,-2.055793,0.039803,0.999831
Mycn,3669.443563,0.243947,0.352910,0.691243,0.489413,0.999831
Rest,5847.264157,-0.146764,0.159514,-0.920066,0.357538,0.999831
Sall1,6139.554237,-0.174725,0.236438,-0.738989,0.459914,0.999831
Tcf7l1,2070.087210,-0.124286,0.235025,-0.528822,0.596929,0.999831
Tcf7l2,178.452354,-0.002236,0.255286,-0.008758,0.993012,0.999831
Tfcp2l1,4497.177490,-0.584142,0.270778,-2.157269,0.030985,0.999831
Tsc22d1,3170.164716,-0.180439,0.287943,-0.626648,0.530890,0.999831
Zfp42,11362.164991,-0.421458,0.227255,-1.854563,0.063659,0.999831


## Combined differential expression in bulk data

In [125]:
reprogramming_de_stats.results_df

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Gnai3,18118.223009,-0.237734,0.046419,-5.121532,3.030633e-07,1.868561e-06
Pbsn,0.000000,NaN,NaN,NaN,NaN,NaN
Cdc45,1046.343936,0.800273,0.094942,8.429116,3.482852e-17,7.204903e-16
H19,4548.672607,0.729453,0.114614,6.364408,1.960450e-10,1.857909e-09
Scml2,183.121673,-0.602971,0.144454,-4.174135,2.991209e-05,1.325471e-04
...,...,...,...,...,...,...
BX571804.1,0.000000,NaN,NaN,NaN,NaN,NaN
AC154773.1,0.000000,NaN,NaN,NaN,NaN,NaN
AL662853.1,0.286036,0.360430,3.104910,0.116084,9.075861e-01,NaN
AC145556.1,0.781459,-1.160610,1.788140,-0.649060,5.162997e-01,NaN


In [135]:
# Combine all results into one dataframe

stats_dict = {
    "Reprogramming": reprogramming_de_stats,
    "Day 2": day_2_de_stats,
    "Day 4": day_4_de_stats,
    "Day 6": day_6_de_stats,
    "Day 10": day_10_de_stats,
    "MEF": mef_de_stats,
    "mESC": esc_de_stats,
    "iPSC": ipsc_de_stats
}

bulk_de = pandas.DataFrame(index=reprogramming_de_stats.results_df.index)
for experiment, stats in stats_dict.items():
    bulk_de[f"{experiment} LFC"] = stats.results_df["log2FoldChange"]
    bulk_de[f"{experiment} p value"] = stats.results_df["pvalue"]
    bulk_de[f"{experiment} p adjusted"] = stats.results_df["padj"]
    bulk_de[f"{experiment} base mean"] = stats.results_df["baseMean"]

lfc_columns = bulk_de.columns.str.contains("LFC")
reduced_columns = ["Reprogramming LFC", "Reprogramming p adjusted", "MEF LFC", "MEF p adjusted", "mESC LFC", "mESC p adjusted", "iPSC LFC", "iPSC p adjusted"]
p_val_columns = bulk_de.columns.str.contains("p value")
p_adj_columns = bulk_de.columns.str.contains("p adjusted")
mean_columns = bulk_de.columns.str.contains("base mean")

In [136]:
# Dead end drivers LFC

bulk_de.loc[dead_end_tfs, reduced_columns]

,Reprogramming LFC,Reprogramming p adjusted,MEF LFC,MEF p adjusted,mESC LFC,mESC p adjusted,iPSC LFC,iPSC p adjusted
Barx2,-1.207086,2.699537e-21,1.025243,0.105690,-0.671832,7.267317e-01,1.424515,0.999831
Dmrtc2,-0.917730,3.002863e-05,-1.048783,NaN,-1.193261,3.935115e-03,-0.047290,0.999831
Hmgb3,-0.857015,2.360299e-15,0.229583,0.345154,-0.439919,2.536387e-04,0.168083,0.999831
Hoxa7,-1.072721,3.138826e-07,0.347961,0.038594,2.789823,1.665563e-04,-1.395663,0.999831
Myc,-0.137470,1.902238e-01,0.144334,0.832980,3.686952,5.225930e-113,-0.569568,0.999831
Ovol1,-1.451013,8.644510e-14,-1.890241,NaN,-0.642798,3.842029e-01,-0.665319,0.999831
Phox2a,-0.313819,3.123642e-04,-1.445185,0.393229,-1.156821,3.216455e-02,-0.356097,0.999831
Tlx2,-1.107785,2.177406e-11,-0.233589,NaN,2.752973,2.657186e-01,-0.921635,0.999831
Tshz1,-0.223540,2.357446e-01,0.259353,0.178275,1.091930,4.040098e-02,-1.179486,0.999831
Zbtb7c,-0.943320,8.875744e-57,-0.638847,0.051838,-0.200119,6.379946e-01,0.265514,0.999831


All except Myc and Tshz1 are significantly downregulated during reprogramming. Most drivers are low in MEFs, stem cells or both so they have high p-values.

Evidence for drivers:
- Barx2: Downregulated in reprogramming, upregulated by Hic2 KO in mESCs but not significantly due to low counts. Could be direct target but there is insufficient evidence.
- Dmrtc2: Downregulated in reprogramming, upregulated by Hic2 KO in mESCs. Probably direct target.
- Hmgb3: Downregulated in reprogramming, upregulated by Hic2 KO in mESCs. Probably direct target.
- Hoxa7: Downregulated in reprogramming, but upregulated in MEFs and downregulated by Hic2 KO in mESCs. Probably not direct target.
- Myc: Downregulated late in reprogramming, but strongly downregulated by Hic2 KO in mESCs. Probably indirect in one of the experiments.
- Ovol1: Downregulated in reprogramming and in the other cells, but too few counts for significance at end points. Could be direct target.
- Phox2a: Downregulated in reprogramming, upregulated by Hic2 KO in mESCs. Probably direct target.
- Tlx2: Downregulated in reprogramming, almost no counts in other cells. Could be direct target, insufficient evidence.
- Tshz1: Upregulated early in reprogramming, downregulated late in reprogramming, upregulated in mESCs. Probably not direct target.
- Zbt7c: Downregulated in reprogramming, downregulated in MEFs and mESCs but not significantly. Could be direct target.

This analysis supports Dmrtc2, Hmgb3 and Phox2a as direct targets, and Barx2, Ovol1, Tlx2 and Zbtb7c could be direct targets.

Hoxa7, Myc and Tshz1 are probably not direct targets based on this analysis.

In [137]:
# Absolute counts for the dead end drivers

bulk_counts[dead_end_tfs]

,Barx2,Dmrtc2,Hmgb3,Hoxa7,Myc,Ovol1,Phox2a,Tlx2,Tshz1,Zbtb7c
Day2_BFP_Rep1,133,58,2030,284,25073,388,255,674,1900,1763
Day2_BFP_Rep2,223,87,2449,357,35224,669,337,780,2856,2551
Day2_BFP_Rep3,102,56,1663,217,23219,336,239,654,1687,1525
Day2_Hic2_Rep1,116,57,2381,407,41371,266,302,496,4547,1598
Day2_Hic2_Rep2,130,45,1863,283,31895,222,219,319,3853,1112
Day2_Hic2_Rep3,78,37,1244,224,23322,140,189,257,2660,844
Day4_BFP_Rep1,355,93,1899,404,30761,1166,257,939,2243,2928
Day4_BFP_Rep1-2,186,45,989,217,17157,603,159,549,1230,1711
Day4_BFP_Rep2,312,164,1919,330,27347,1056,234,994,1569,2222
Day4_BFP_Rep3,350,116,2084,434,32940,1293,288,1110,2090,2872


In [138]:
# Differential expression of reprogramming drivers

bulk_de.loc[stem_cell_tfs, reduced_columns]

,Reprogramming LFC,Reprogramming p adjusted,MEF LFC,MEF p adjusted,mESC LFC,mESC p adjusted,iPSC LFC,iPSC p adjusted
Jarid2,0.820343,1.214135e-08,0.214631,0.401339,0.159998,3.127053e-01,-0.088999,0.999831
Klf2,-0.180294,5.962700e-01,-0.859362,0.072763,0.280519,3.352921e-02,-0.417831,0.999831
Mycn,0.170911,4.827383e-01,-0.169110,0.591767,-1.605242,3.891848e-07,0.243947,0.999831
Rest,0.492894,6.121253e-05,-0.011507,0.962355,0.464189,1.023140e-08,-0.146764,0.999831
Sall1,1.516870,8.706573e-22,0.269536,0.252543,-0.270905,9.704719e-04,-0.174725,0.999831
Tcf7l1,1.263931,2.177080e-13,0.255627,0.321822,0.484475,5.611239e-03,-0.124286,0.999831
Tcf7l2,0.753992,1.340578e-09,0.411018,0.026791,0.314795,1.234990e-01,-0.002236,0.999831
Tfcp2l1,1.502553,2.759063e-10,-0.552028,0.152600,2.041081,7.917192e-39,-0.584142,0.999831
Tsc22d1,0.063317,7.309035e-01,-0.015511,0.939704,0.545665,8.381747e-02,-0.180439,0.999831
Zfp42,1.344900,6.666513e-18,0.266531,0.847692,1.464963,1.111372e-28,-0.421458,0.999831


In [139]:
# Absolute counts for stem cell drivers

bulk_counts[stem_cell_tfs]

,Jarid2,Klf2,Mycn,Rest,Sall1,Tcf7l1,Tcf7l2,Tfcp2l1,Tsc22d1,Zfp42
Day2_BFP_Rep1,1538,19,39,2638,190,315,907,30,2491,89
Day2_BFP_Rep2,2185,47,55,3534,321,500,1303,44,3724,137
Day2_BFP_Rep3,1275,9,21,2409,177,279,764,43,1854,103
Day2_Hic2_Rep1,2945,35,56,4665,740,820,1545,89,3760,220
Day2_Hic2_Rep2,2325,26,51,3688,628,591,1413,62,2849,228
Day2_Hic2_Rep3,1769,11,39,2992,400,451,1152,42,2027,129
Day4_BFP_Rep1,2205,23,51,2895,513,331,1657,110,3365,224
Day4_BFP_Rep1-2,1269,10,23,1617,279,167,895,59,1881,136
Day4_BFP_Rep2,1787,51,26,1976,379,201,998,108,1979,270
Day4_BFP_Rep3,2156,39,50,2810,478,334,1532,138,3472,248


## Save differential expression results

In [140]:
bulk_de.to_csv("RNA Sequencing Data/bulk_differential_expression.csv")
metadata_df.to_csv("RNA Sequencing Data/bulk_sequencing_metadata.csv")

# Differential expression with SCVI-tools